In [7]:
%load_ext autoreload



In [8]:
%autoreload 2

In [2]:

from wearable_affect.data import DEFAULT_DATA_DIR, PROJECT_ROOT, list_subjects, load_subject
from wearable_affect.windows import make_windows
from wearable_affect.features import build_feature_table

windows = []
for sid in list_subjects(DEFAULT_DATA_DIR):
    windows.extend(make_windows(load_subject(DEFAULT_DATA_DIR, sid)))

features = build_feature_table(windows)
print(features.shape)
features.head()

(1015, 17)


,subject_id,start_s,target,scl_mean,scl_slope,phasic_std,scr_count,scr_amplitude_mean,temp_mean,temp_std,temp_slope,acc_mean,acc_std,hr_mean,hr_std,hrv_sdnn,hrv_rmssd
0,S10,120,0,0.383118,-0.000468,0.002717,4.0,0.011513,33.327126,0.011239,0.000238,62.994370,1.542139,92.126861,26.119334,253.821753,315.595379
1,S10,150,0,0.378540,0.000031,0.003311,4.0,0.016309,33.330250,0.008656,0.000049,62.960930,1.260934,92.867746,21.714411,262.674365,303.188650
2,S10,180,0,0.378806,-0.000052,0.003031,4.0,0.015350,33.324543,0.011929,-0.000297,62.891094,1.930000,95.046911,20.144091,258.721070,186.482827
3,S10,210,0,0.377676,-0.000043,0.001908,1.0,0.010234,33.322254,0.012108,0.000215,62.782509,1.491622,101.626880,13.480114,129.890619,152.643935
4,S10,240,0,0.380693,0.000251,0.004739,3.0,0.029847,33.331497,0.015091,0.000492,62.742710,0.380364,96.955855,15.905584,165.820024,165.732199


In [3]:
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
features.to_parquet(out_dir / "features.parquet", index=False)

In [4]:
features.groupby("target")[["scl_mean", "scl_slope", "scr_count", "hr_mean", "temp_slope"]].mean()

,scl_mean,scl_slope,scr_count,hr_mean,temp_slope
target,,,,,
0,1.339609,-0.000948,7.249649,75.696366,0.000399
1,3.439927,0.001186,19.870861,88.704947,-0.001141


In [ ]:
# from previous run; retained for comparison
features.groupby("target")[["scl_mean", "scl_slope", "scr_count", "hr_mean", "temp_slope"]].mean()

,scl_mean,scl_slope,scr_count,hr_mean,temp_slope
target,,,,,
0,1.339609,-0.000948,40.450210,75.696366,0.000399
1,3.439927,0.001186,20.728477,88.704947,-0.001141


In [1]:
import datetime
import inspect
import wearable_affect.features as f

print(f.__file__)
print("has new threshold:", hasattr(f, "SCR_MIN_AMPLITUDE_US"))
print(inspect.getsource(f.eda_features))

p = PROJECT_ROOT / "data" / "processed" / "features.parquet"
print("parquet last saved:", datetime.datetime.fromtimestamp(p.stat().st_mtime))

C:\Users\maheshbh\wearable-affect\src\wearable_affect\features.py
has new threshold: True
def eda_features(eda: np.ndarray, fs: int) -> dict[str, float]:
    """Tonic level (SCL) and phasic responses (SCRs) from an EDA segment."""
    decomposed = nk.eda_phasic(eda, sampling_rate=fs, method="smoothmedian")
    tonic = decomposed["EDA_Tonic"].to_numpy()
    phasic = decomposed["EDA_Phasic"].to_numpy()
    try:
        # amplitude_min=0 turns off NeuroKit2's *relative* threshold; we apply an absolute one below.
        _, info = nk.eda_peaks(phasic, sampling_rate=fs, amplitude_min=0)
        amplitudes = np.asarray(info["SCR_Amplitude"], dtype=float)
    except Exception:
        # No detectable responses (e.g. a flat segment) counts as zero SCRs.
        amplitudes = np.array([])
    amplitudes = amplitudes[amplitudes >= SCR_MIN_AMPLITUDE_US]  # also drops NaN


    return {
        "scl_mean": float(np.mean(tonic)),
        "scl_slope": _slope(tonic, fs),
        "phasic_std": float(np

NameError: name 'PROJECT_ROOT' is not defined